# ✈️ Flight Price Prediction

# Notebook 04: Feature Engineering

## Objective

The purpose of this notebook is to transform raw features into meaningful numerical features that improve model performance.

### Tasks
- Extract features from journey date
- Extract features from departure time
- Extract features from arrival time
- Convert flight duration into total minutes
- Convert total stops into numerical values
- Remove redundant columns
- Save the feature-engineered dataset

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
DATA_DIR = Path("../data/processed")

df = pd.read_csv(DATA_DIR / "cleaned_train.csv")

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (10462, 10)


,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,13302


In [3]:
# Convert journey date into datetime format
df["Date_of_Journey"] = pd.to_datetime(
    df["Date_of_Journey"],
    format="%d/%m/%Y"
)

# Extract useful features
df["Journey_Day"] = df["Date_of_Journey"].dt.day
df["Journey_Month"] = df["Date_of_Journey"].dt.month
df["Journey_Weekday"] = df["Date_of_Journey"].dt.dayofweek

df[
    [
        "Date_of_Journey",
        "Journey_Day",
        "Journey_Month",
        "Journey_Weekday"
    ]
].head()

,Date_of_Journey,Journey_Day,Journey_Month,Journey_Weekday
0,2019-03-24,24,3,6
1,2019-05-01,1,5,2
2,2019-06-09,9,6,6
3,2019-05-12,12,5,6
4,2019-03-01,1,3,4


In [4]:
df["Dep_Time"] = pd.to_datetime(
    df["Dep_Time"],
    format="%H:%M"
)

df["Departure_Hour"] = df["Dep_Time"].dt.hour
df["Departure_Minute"] = df["Dep_Time"].dt.minute

df[
    [
        "Dep_Time",
        "Departure_Hour",
        "Departure_Minute"
    ]
].head()

,Dep_Time,Departure_Hour,Departure_Minute
0,1900-01-01 22:20:00,22,20
1,1900-01-01 05:50:00,5,50
2,1900-01-01 09:25:00,9,25
3,1900-01-01 18:05:00,18,5
4,1900-01-01 16:50:00,16,50


In [5]:
df["Arrival_Time"] = pd.to_datetime(
    df["Arrival_Time"].str.split(" ").str[0],
    format="%H:%M"
)

df["Arrival_Hour"] = df["Arrival_Time"].dt.hour
df["Arrival_Minute"] = df["Arrival_Time"].dt.minute

df[
    [
        "Arrival_Time",
        "Arrival_Hour",
        "Arrival_Minute"
    ]
].head()

,Arrival_Time,Arrival_Hour,Arrival_Minute
0,1900-01-01 01:10:00,1,10
1,1900-01-01 13:15:00,13,15
2,1900-01-01 04:25:00,4,25
3,1900-01-01 23:30:00,23,30
4,1900-01-01 21:35:00,21,35


In [6]:
def convert_duration(duration):
    """
    Convert duration strings into total minutes.

    Examples:
        2h 50m -> 170
        5h     -> 300
        45m    -> 45
    """

    duration = duration.strip()

    hours = 0
    minutes = 0

    if "h" in duration:
        hours = int(duration.split("h")[0])

    if "m" in duration:
        if "h" in duration:
            minute_part = duration.split("h")[1].replace("m", "").strip()
            minutes = int(minute_part) if minute_part else 0
        else:
            minutes = int(duration.replace("m", "").strip())

    return hours * 60 + minutes


df["Duration_Minutes"] = df["Duration"].apply(convert_duration)

df[
    [
        "Duration",
        "Duration_Minutes"
    ]
].head()

,Duration,Duration_Minutes
0,2h 50m,170
1,7h 25m,445
2,19h,1140
3,5h 25m,325
4,4h 45m,285


In [7]:
df["Total_Stops"] = (
    df["Total_Stops"]
    .replace("non-stop", "0 stops")
    .str.extract(r"(\d+)")
    .astype(int)
)

df[
    [
        "Total_Stops"
    ]
].head()

,Total_Stops
0,0
1,2
2,2
3,1
4,1


In [8]:
columns_to_drop = [
    "Date_of_Journey",
    "Dep_Time",
    "Arrival_Time",
    "Duration",
    "Route"
]

df.drop(columns=columns_to_drop, inplace=True)

df.head()

,Airline,Source,Destination,Total_Stops,Price,Journey_Day,Journey_Month,Journey_Weekday,Departure_Hour,Departure_Minute,Arrival_Hour,Arrival_Minute,Duration_Minutes
0,IndiGo,Banglore,New Delhi,0,3897,24,3,6,22,20,1,10,170
1,Air India,Kolkata,Banglore,2,7662,1,5,2,5,50,13,15,445
2,Jet Airways,Delhi,Cochin,2,13882,9,6,6,9,25,4,25,1140
3,IndiGo,Kolkata,Banglore,1,6218,12,5,6,18,5,23,30,325
4,IndiGo,Banglore,New Delhi,1,13302,1,3,4,16,50,21,35,285


In [9]:
print("=" * 50)
print("Feature Engineering Completed")
print("=" * 50)

print(f"\nDataset Shape : {df.shape}")

print("\nColumns\n")
print(df.columns.tolist())

print("\nMissing Values\n")
print(df.isnull().sum())

Feature Engineering Completed

Dataset Shape : (10462, 13)

Columns

['Airline', 'Source', 'Destination', 'Total_Stops', 'Price', 'Journey_Day', 'Journey_Month', 'Journey_Weekday', 'Departure_Hour', 'Departure_Minute', 'Arrival_Hour', 'Arrival_Minute', 'Duration_Minutes']

Missing Values

Airline             0
Source              0
Destination         0
Total_Stops         0
Price               0
Journey_Day         0
Journey_Month       0
Journey_Weekday     0
Departure_Hour      0
Departure_Minute    0
Arrival_Hour        0
Arrival_Minute      0
Duration_Minutes    0
dtype: int64


In [10]:
engineered_features = [
    "Journey_Day",
    "Journey_Month",
    "Journey_Weekday",
    "Departure_Hour",
    "Departure_Minute",
    "Arrival_Hour",
    "Arrival_Minute",
    "Duration_Minutes",
    "Total_Stops"
]

df[engineered_features].head()

,Journey_Day,Journey_Month,Journey_Weekday,Departure_Hour,Departure_Minute,Arrival_Hour,Arrival_Minute,Duration_Minutes,Total_Stops
0,24,3,6,22,20,1,10,170,0
1,1,5,2,5,50,13,15,445,2
2,9,6,6,9,25,4,25,1140,2
3,12,5,6,18,5,23,30,325,1
4,1,3,4,16,50,21,35,285,1


In [11]:
OUTPUT_DIR = Path("../data/processed")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT_DIR / "featured_train.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!


# Summary

### New Features Created

- Journey_Day
- Journey_Month
- Journey_Weekday
- Departure_Hour
- Departure_Minute
- Arrival_Hour
- Arrival_Minute
- Duration_Minutes

### Modified Feature

- Total_Stops → Numerical

### Removed Features

- Date_of_Journey
- Dep_Time
- Arrival_Time
- Duration
- Route

The dataset is now ready for preprocessing and model training.